# AGESC stationnary Ekman model (Morales-Márquez 2021)


In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250, add_mask_inside_swot, build_swath_polygon

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature


import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

_________
# ERA5 data

In [2]:
era5 = ['/Users/mdemol/DATA_WIND/era5/2023_waves_interp.nc',
        '/Users/mdemol/DATA_WIND/era5/adaptor.mars.internal-1726002205.1540956-13738-7-c8d88fd1-3ec7-4113-8790-c92c59938aa6.nc',#u10 etc
        #'/Users/mdemol/DATA_WIND/era5/adaptor.mars.internal-1725888040.2442634-4062-7-717f5d1f-4dbc-4297-ba65-42766024d861.nc' # lon, lat wrong coords
        #'/Users/mdemol/DATA_WIND/era5/corrected_coords_d68.nc',# corrected coords
        '/Users/mdemol/DATA_WIND/era5/corrected_coords_d68_interp.nc'#ust, vst etc
       ]
era = xr.open_mfdataset(era5)
era['f'] = 2 * 2 * np.pi / 86164.1 * np.sin(era.latitude * np.pi / 180)
era = era.sel(longitude = slice(0, 15), latitude = slice(44, 36), time = slice(pd.to_datetime('2023-03-01T00:00:00'), pd.to_datetime('2023-07-31T23:00:00')))

era['u10'] = era['u10'].assign_attrs({'long_name':r'Zonal wind velocity at 10m $u_{10}$', 'units':'m/s'})
era['v10'] = era['v10'].assign_attrs({'long_name':r'Meridional wind velocity at 10m $v_{10}$', 'units':'m/s'})

In [3]:
era 

<xarray.Dataset> Size: 2GB
Dimensions:    (longitude: 61, latitude: 33, time: 3672)
Coordinates:
  * longitude  (longitude) float32 244B 0.0 0.25 0.5 0.75 ... 14.5 14.75 15.0
  * latitude   (latitude) float32 132B 44.0 43.75 43.5 43.25 ... 36.5 36.25 36.0
  * time       (time) datetime64[ns] 29kB 2023-03-01 ... 2023-07-31T23:00:00
Data variables: (12/27)
    u10        (time, latitude, longitude) float64 59MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    v10        (time, latitude, longitude) float64 59MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    t2m        (time, latitude, longitude) float64 59MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    ewss       (time, latitude, longitude) float64 59MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    iews       (time, latitude, longitude) float64 59MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    inss       (time, latitude, longitude) float64 59MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    ...         ...
    swh        (time, latitude, longitude) float64 59MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    shts       (time, latitude, longitude) float64 59MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    ust        (time, latitude, longitude) float64 59MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    vst        (time, latitude, longitude) float64 59MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    mwp        (time, latitude, longitude) float32 30MB dask.array<chunksize=(3672, 33, 61), meta=np.ndarray>
    f          (latitude) float32 132B 0.0001013 0.0001009 ... 8.572e-05
Attributes:
    Conventions:  CF-1.6
    history:      2024-09-10 21:05:21 GMT by grib_to_netcdf-2.28.1: /opt/ecmw...

__________
# Apply AGESC model
## For detailed description : 
Regionalizing the Impacts of Wind- and Wave-Induced Currents on Surface Ocean Dynamics: A Long-Term Variability Analysis in the Mediterranean Sea, Morales-Márquez 2021, https://onlinelibrary.wiley.com/doi/abs/10.1029/2020JC017104

## Summary

### Main assumptions
- Constant drag coefficient $A_z$
- monochromatic wavefield propagating in deep waters
### Constants and notations (vectors are in complex notation)
- Sea water density $\rho_w = 1.03$kg/m3
- Air density $\rho_a=1.2$kg/m3
- The gravity $g=9.81$m.s$^{-2}$
- Wind velocity at 10m $\vec{u}_{10}$ m/s and its norm $U_{10}$ (from ERA5)
- Neutral drag coefficient $C_D= (2.7/u_{10} + 0.142 +0.0764 u_{10})/1000 $
- Turbulent stress $\vec{\tau} = \rho_a C_D u_{10}\vec{u_{10}}$ N.m$^{-2}$
- Vertical eddy viscosity $A_z = 1.07 10^{-2}$ $m^2.s^{-1}$
- The Coriolis frequency $f$
- The characteritic Ekman layer depth $\delta = 1/m$ with $m= (1+i)\sqrt{f/2A_z}$
- The characteritic Stokes depth $\delta_s = 1/k$ with $k= \omega^2/g$
- The wave amplitude $a$ (from ERA5)
- The wave direction $\theta_w$ (from ERA5)
- The wave frequency $\omega$ ($2\pi/T$ from ERA5)
  
### Four ageostrophic currents components
$$ \vec{U_a}(z) = \vec{U}(z) +  \vec{U_{\tau_z}}(z) +\vec{U_S}(z)+ \vec{U_{ES}}(z) $$

- The classical Ekman component : $$ \vec{U}(z) = \frac{\vec{\tau}}{\rho_w A_z m}e^{mz} $$

  
- The current induced by the wave radiation stress : $$\vec{U_{\tau_z}}(z) = \frac{\partial_X S}{\rho_w A_z m }e^{mz}$$
  with $\partial_XS = [\partial_x S_{xx} + \partial_y S_{yx}] + i[\partial_x S_{xy} + \partial_y S_{yy}]$, $S_{xx} = E/2 cos^2(\theta_{\omega})$, $S_{xy} = S_{yx} = E/2 sin(\theta_{\omega}) cos(\theta_{\omega})$, $S_{yy} = E/2 sin^2(\theta_{\omega})$ with $E=\rho_w g a^2/2$

  
-  The stokes component = decreases over $\delta_s$, correlated with the dynamical response to the Coriolis-Stokes force :
$$\vec{U_S}(z) = \frac{m^2 U_{S0}}{4k^2 - m^2}e^{2kz}$$
with $U_{S0} = a^2 \omega k (cos(\theta_{\omega}) + i sin(\theta_{\omega})$


-  The Ekman-Stokes component = non linear interaction between wind and waves acting over the entire Ekman layer :
$$\vec{U_{ES}}(z) = - \frac{2 k m U_{S0}}{4k^2 -m^2}e^{mz}$$


# Estimates of the vertical turbulent stress divergence

$$ \frac{1}{\rho_w}\partial_z \vec{\tau} = \partial_z(A_z \partial_z \vec{U}_a) = i f(\vec{U}_a + \vec{U}_s) + \vec{T}_{uds} $$

with $$ \vec{U}_s = a^2 \omega k e^{2kz} $$

In [4]:
def add_agesc_currents(z, era) :
    if z>0 :
        print('WARNING : z should be negative')
    era=era.copy()    
    #vars
    rhow = 1.03*1e3 #kg/m3
    rhoa = 1.2#kg/m3
    g = 9.81#m/s2
    a = era.swh#m
    U10 = era.u10 + 1j*era.v10#m/s
    u10 = np.sqrt(np.conj(U10)*U10)#m/s
    Cd = (2.7/u10 + 0.142 +0.0764*u10)/1e3
    R = 6378e3 #m
    Az = 1.07e-2#m2/s
    #Az = 1e-3#m2/s
    m=(1+1j)*np.sqrt(era.f/(2*Az))#/m
    tau = rhoa * Cd *u10 *U10
    #tau = era.iews + 1j *era.inss # N/m2
    E = rhow*g*a**2/2 #m.kg/s2 = N
    thetaw = era.mwd*np.pi/180 #rad
    omega = 2*np.pi/era.mwp #rad/s
    k = omega**2/g#/m
    #K = k*np.exp(1j*era.mwd*np.pi/180)
    
    # ue classical Ekman
    ue = tau/(rhow*Az*m) * np.exp(m*z)

    # utauz
    Sxx = E/2*np.cos(thetaw)**2
    Sxy = E/2*np.cos(thetaw)*np.sin(thetaw)
    Syx = Sxy
    Syy = E/2*np.cos(thetaw)**2
    deglat_km = 111111 #m/deglat
    dlon_dx = 1/(deglat_km * np.cos(era.latitude * np.pi / 180))
    dlat_dy = 1/deglat_km
    partialS = Sxx.differentiate('longitude')*dlon_dx + Sxy.differentiate('latitude')*dlat_dy + 1j*(Sxy.differentiate('longitude')*dlon_dx + Syy.differentiate('latitude')*dlat_dy)
    utauz = partialS/(rhow*Az*m) * np.exp(m*z)

    # us 
    us0 = a**2*omega*k*np.exp(1j*thetaw)
    us = (m**2 * us0) /(4*k**2 -m**2) *np.exp(2*k*z)

    #ues
    ues = - 2*k*m*us0/(4*k**2 - m**2)*np.exp(m*z)

    #in era
    mo = '_agesc'
    era['ue'+mo], era['ve'+mo] = ue.real.assign_attrs({'long_name':r'Zonal AGESC $u_{Ekman}$', 'units':'m/s'}), ue.imag.assign_attrs({'long_name':r'Meridional AGESC $v_{Ekman}$', 'units':'m/s'})
    era['utauz'+mo], era['vtauz'+mo] = utauz.real.assign_attrs({'long_name':r'Zonal AGESC $u_{\tau_z}$', 'units':'m/s'}), utauz.imag.assign_attrs({'long_name':r'Meridional AGESC $v_{\tau_z}$', 'units':'m/s'})
    era['us'+mo], era['vs'+mo] = us.real.assign_attrs({'long_name':r'Zonal AGESC $u_{Stokes}$', 'units':'m/s'}), us.imag.assign_attrs({'long_name':r'Meridional AGESC $v_{Stokes}$', 'units':'m/s'})
    era['ues'+mo], era['ves'+mo] = ues.real.assign_attrs({'long_name':r'Zonal AGESC $u_{EkmanStokes}$', 'units':'m/s'}), ues.imag.assign_attrs({'long_name':r'Meridional AGESC $v_{EkmanStokes}$', 'units':'m/s'})
    era['us0'+mo], era['vs0'+mo] = us0.real.assign_attrs({'long_name':r'Zonal AGESC $u_{EkmanStokes}(z=0)$', 'units':'m/s'}), us0.imag.assign_attrs({'long_name':r'Meridional AGESC $v_{EkmanStokes}(z=0)$', 'units':'m/s'})

    # Total 
    era['ua'+mo] = (ue +utauz + us + ues).real.assign_attrs({'long_name':r'Zonal AGESC $u_{ageo}$', 'units':'m/s'})
    era['va'+mo] = (ue +utauz + us + ues).imag.assign_attrs({'long_name':r'Meridional AGESC $u_{ageo}$', 'units':'m/s'})

    # Waves 
    era['uw'+mo] = (utauz + us + ues).real.assign_attrs({'long_name':r'Zonal AGESC $u_{waves}$', 'units':'m/s'})
    era['vw'+mo] = (utauz + us + ues).imag.assign_attrs({'long_name':r'Meridional AGESC $v_{waves}$', 'units':'m/s'})


    # Stokes
    era['ustokes'+mo] = (a**2*omega*k*np.exp(2*k*z)).real.assign_attrs({'long_name':r'Zonal AGESC $u_{s}$', 'units':'m/s'})
    era['vstokes'+mo] = (a**2*omega*k*np.exp(2*k*z)).imag.assign_attrs({'long_name':r'Meridional AGESC $v_{s}$', 'units':'m/s'})

    # Vertical turbulent Stress divergence
    era['vsde'+mo] = - era.f * (era['va'+mo]+era['vs'+mo]).assign_attrs({'long_name':r'Zonal AGESC VSD', 'units':r'$m.s^{-2}$'})
    era['vsdn'+mo] = era.f * (era['ua'+mo]+era['us'+mo]).assign_attrs({'long_name':r'Meridional AGESC VSD', 'units':r'$m.s^{-2}$'})
    
    #Ekman rio
    if z==0 :
        theta0 = - 22.5 * np.pi / 180 * np.sign(era.f)
        beta0 = 0.6
        era['ue_rioold'] = (beta0 * (np.cos(theta0) * era.iews - np.sin(theta0) * era.inss)).assign_attrs({'long_name':r'Zonal Rio $u_{Ekman}$', 'units':'m/s'})
        era['ve_rioold'] = (beta0 * (np.sin(theta0) * era.iews + np.cos(theta0) * era.inss)).assign_attrs({'long_name':r'Meridional Rio $v_{Ekman}$', 'units':'m/s'})

        theta0 = - 33.125 * np.pi / 180 * np.sign(era.f)
        beta0 = 1.08
        era['ue_rio'] = (beta0 * (np.cos(theta0) * era.iews - np.sin(theta0) * era.inss)).assign_attrs({'long_name':r'Zonal updated Rio $u_{Ekman}$', 'units':'m/s'})
        era['ve_rio'] = (beta0 * (np.sin(theta0) * era.iews + np.cos(theta0) * era.inss)).assign_attrs({'long_name':r'Meridional updated Rio $v_{Ekman}$', 'units':'m/s'})

    if z==-15 :
        theta15 = - 37.5 * np.pi / 180 * np.sign(era.f)#adapted for mediterranean sea
        beta15 = 0.15
        era['ue_rioold'] = (beta15 * (np.cos(theta15) * era.iews - np.sin(theta15) * era.inss)).assign_attrs({'long_name':r'Zonal Rio $u_{Ekman}$', 'units':'m/s'})
        era['ve_rioold'] = (beta15 * (np.sin(theta15) * era.iews + np.cos(theta15) * era.inss)).assign_attrs({'long_name':r'Meridional Rio $v_{Ekman}$', 'units':'m/s'})
            
        theta15 = - 62.5 * np.pi / 180 * np.sign(era.f)#adapted for mediterranean sea
        beta15 = 0.35
        era['ue_rio'] = (beta15 * (np.cos(theta15) * era.iews - np.sin(theta15) * era.inss)).assign_attrs({'long_name':r'Zonal updated Rio $u_{Ekman}$', 'units':'m/s'})
        era['ve_rio'] = (beta15 * (np.sin(theta15) * era.iews + np.cos(theta15) * era.inss)).assign_attrs({'long_name':r'Meridional updated Rio $v_{Ekman}$', 'units':'m/s'})

    # Vertical turbulent Stress divergence
    era['vsde_rio'] = - era.f * (era.ve_rio).assign_attrs({'long_name':r'Zonal updated Rio VSD', 'units':r'$m.s^{-2}$'})
    era['vsdn_rio'] = era.f * (era.ue_rio).assign_attrs({'long_name':r'Meridional updated Rio VSD', 'units':r'$m.s^{-2}$'})

    # Vertical turbulent Stress divergence
    era['vsde_rioold'] = - era.f * (era.ve_rioold).assign_attrs({'long_name':r'Zonal updated Rio VSD', 'units':r'$m.s^{-2}$'})
    era['vsdn_rioold'] = era.f * (era.ue_rioold).assign_attrs({'long_name':r'Meridional updated Rio VSD', 'units':r'$m.s^{-2}$'})
    
    # Norms
    for v in ['ve'+mo, 'vtauz'+mo, 'vs'+mo, 'ves'+mo, 've_rio','ve_rioold', 'va'+mo, 'vw'+mo, 'v10'] :
        era[v.replace('v', 'U')] = np.sqrt(era[v.replace('v', 'u')]**2 + era[v]**2).assign_attrs({'long_name':era[v].attrs['long_name'].replace('Meridional', '').replace('v_','U_'), 'units':'m/s'})
   
    return era

era0 = add_agesc_currents(0, era)
era15 = add_agesc_currents(-15, era)  

In [5]:
list(era0.keys())

['u10',
 'v10',
 't2m',
 'ewss',
 'iews',
 'inss',
 'msl',
 'nsss',
 'sst',
 'ssr',
 'ssrc',
 'str',
 'strc',
 'sp',
 'lgws',
 'zust',
 'lsm',
 'mwd',
 'tauoc',
 'mgws',
 'pp1d',
 'swh',
 'shts',
 'ust',
 'vst',
 'mwp',
 'f',
 'ue_agesc',
 've_agesc',
 'utauz_agesc',
 'vtauz_agesc',
 'us_agesc',
 'vs_agesc',
 'ues_agesc',
 'ves_agesc',
 'us0_agesc',
 'vs0_agesc',
 'ua_agesc',
 'va_agesc',
 'uw_agesc',
 'vw_agesc',
 'ustokes_agesc',
 'vstokes_agesc',
 'vsde_agesc',
 'vsdn_agesc',
 'ue_rioold',
 've_rioold',
 'ue_rio',
 've_rio',
 'vsde_rio',
 'vsdn_rio',
 'vsde_rioold',
 'vsdn_rioold',
 'Ue_agesc',
 'Utauz_agesc',
 'Us_agesc',
 'Ues_agesc',
 'Ue_rio',
 'Ue_rioold',
 'Ua_agesc',
 'Uw_agesc',
 'U10']

In [6]:
#rename vars
vars_wind = [v for v in era0 if 'agesc' in v] + [v for v in era0 if 'rio' in v]


era0.rename({v: v+'_z0' for v in vars_wind}).to_netcdf(os.path.join('/Users/mdemol/DATA_WIND/era5', 'era5_agesc_rio_z0.nc'))
era15.rename({v: v+'_z15' for v in vars_wind}).to_netcdf(os.path.join('/Users/mdemol/DATA_WIND/era5', 'era5_agesc_rio_z15.nc'))

/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/dask/core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


In [14]:
vars_wind

['ue_agesc',
 've_agesc',
 'utauz_agesc',
 'vtauz_agesc',
 'us_agesc',
 'vs_agesc',
 'ues_agesc',
 'ves_agesc',
 'us0_agesc',
 'vs0_agesc',
 'ua_agesc',
 'va_agesc',
 'uw_agesc',
 'vw_agesc',
 'ustokes_agesc',
 'vstokes_agesc',
 'vsde_agesc',
 'vsdn_agesc',
 'Ue_agesc',
 'Utauz_agesc',
 'Us_agesc',
 'Ues_agesc',
 'Ua_agesc',
 'Uw_agesc',
 'ue_rioold',
 've_rioold',
 'ue_rio',
 've_rio',
 'vsde_rio',
 'vsdn_rio',
 'vsde_rioold',
 'vsdn_rioold',
 'Ue_rio',
 'Ue_rioold']